# Symptom checker — disease/condition classifier

**Dataset:** Disease Prediction Using Machine Learning — Kaggle
Source: https://www.kaggle.com/datasets/kaushil268/disease-prediction-using-machine-learning

Two files: `Training.csv` and `Testing.csv`. 132 binary symptom columns (0/1), final
column `prognosis` = one of 42 disease labels.

**Important caveat to put in your report:** this dataset's symptom-to-disease mappings are
generic/global, not Nepal-specific or clinically validated against Nepali epidemiology. We
treat its output as a coarse triage signal only — pointing the user toward a medicine
*category* and a "see a doctor" urgency flag — never as a diagnosis. The model's predicted
condition is cross-checked against the Department of Drug Administration (DDA) Nepal
Essential Drug List so only conditions with a realistic, locally available treatment path
are surfaced to the user.

**What this notebook does:**
1. Loads the real Kaggle symptom/disease dataset
2. Trains a classifier (Random Forest — chosen for interpretability via feature importances,
   which matters when you're asked "how does it decide" in a viva)
3. Evaluates with accuracy, F1, and per-class precision/recall
4. Inspects feature importances (which symptoms matter most)
5. Exports the trained model plus the ordered symptom list Django/Flutter need to build the
   checkbox UI

**Setup before running:**
- Download `Training.csv` and `Testing.csv` from the Kaggle link above
- Place both into `./data/`


In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

DATA_DIR = Path("../datasets")
OUTPUT_DIR = Path("../models")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## 1. Load the real dataset

In [2]:
train_df = pd.read_csv(DATA_DIR / "Training.csv")
test_df = pd.read_csv(DATA_DIR / "Testing.csv")

# Some versions of this dataset ship an extra unnamed trailing column — drop it if present
train_df = train_df.loc[:, ~train_df.columns.str.contains("^Unnamed")]
test_df = test_df.loc[:, ~test_df.columns.str.contains("^Unnamed")]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print()
print("Number of symptom columns:", train_df.shape[1] - 1)
print("Number of unique diseases:", train_df["prognosis"].nunique())
train_df.head(3)

Train shape: (4920, 133)
Test shape: (42, 133)

Number of symptom columns: 132
Number of unique diseases: 41


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [3]:
# Sanity checks
print("Missing values (train):", train_df.isnull().sum().sum())
print("Missing values (test):", test_df.isnull().sum().sum())
print()
print("Sample of disease labels:")
print(sorted(train_df["prognosis"].unique())[:15])
print()
print("Class balance (train) — top 10:")
print(train_df["prognosis"].value_counts().head(10))

Missing values (train): 0
Missing values (test): 0

Sample of disease labels:
['(vertigo) Paroymsal  Positional Vertigo', 'AIDS', 'Acne', 'Alcoholic hepatitis', 'Allergy', 'Arthritis', 'Bronchial Asthma', 'Cervical spondylosis', 'Chicken pox', 'Chronic cholestasis', 'Common Cold', 'Dengue', 'Diabetes ', 'Dimorphic hemmorhoids(piles)', 'Drug Reaction']

Class balance (train) — top 10:
prognosis
Fungal infection       120
Allergy                120
GERD                   120
Chronic cholestasis    120
Drug Reaction          120
Peptic ulcer diseae    120
AIDS                   120
Diabetes               120
Gastroenteritis        120
Bronchial Asthma       120
Name: count, dtype: int64


## 2. Split features and target

Symptom columns are already binary (0/1) — no encoding needed. We keep the full 132-symptom
feature space for training (better accuracy), but separately export a *trimmed* list of the
~40-50 most common/relevant symptoms for the actual checkbox UI in the app, since asking a
user to scroll through 132 checkboxes is bad UX. The model still accepts the full vector at
inference time — unselected symptoms are just left at 0.

In [4]:
symptom_columns = [c for c in train_df.columns if c != "prognosis"]

X_train = train_df[symptom_columns]
y_train = train_df["prognosis"]
X_test = test_df[symptom_columns]
y_test = test_df["prognosis"]

print("Feature columns:", len(symptom_columns))
print("Example symptom names:", symptom_columns[:10])

Feature columns: 132
Example symptom names: ['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing', 'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity', 'ulcers_on_tongue']


## 3. Train the classifier

Random Forest — handles binary features natively, gives us feature importances for free
(useful to discuss in a viva), and performs well on datasets like this where the
symptom -> disease mapping is fairly deterministic (each disease tends to have a near-unique
symptom signature in this particular dataset).

In [5]:
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)

clf.fit(X_train, y_train)

# 5-fold cross-validation on the training set, to report a more robust accuracy estimate
# than a single train/test split alone
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="accuracy")
print("5-fold CV accuracy: %.4f (+/- %.4f)" % (cv_scores.mean(), cv_scores.std()))

5-fold CV accuracy: 1.0000 (+/- 0.0000)


In [6]:
pred = clf.predict(X_test)

print("=== Held-out test set performance ===")
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Macro F1:", round(f1_score(y_test, pred, average="macro"), 4))
print()
print(classification_report(y_test, pred, zero_division=0))

=== Held-out test set performance ===
Accuracy: 0.9762
Macro F1: 0.9837

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       0.50      1.00      0.67         1
                    Chronic cholestasis       1.00      1.00      1.00         1
                            Common 

In [7]:
# Confusion matrix — with 42 classes this is large, so we just check the diagonal
# (correct predictions) rather than print the full matrix
cm = confusion_matrix(y_test, pred, labels=sorted(y_train.unique()))
diagonal_correct = np.trace(cm)
total = cm.sum()
print(f"Correct predictions: {diagonal_correct} / {total} ({100*diagonal_correct/total:.2f}%)")

# Show any diseases the model confuses with each other (off-diagonal entries > 0)
labels_sorted = sorted(y_train.unique())
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
confusions = []
for actual in labels_sorted:
    for predicted in labels_sorted:
        if actual != predicted and cm_df.loc[actual, predicted] > 0:
            confusions.append((actual, predicted, cm_df.loc[actual, predicted]))

if confusions:
    print()
    print("Misclassifications (actual -> predicted, count):")
    for a, p, c in confusions:
        print(f"  {a} -> {p}: {c}")
else:
    print()
    print("No misclassifications on the held-out test set.")

Correct predictions: 41 / 42 (97.62%)

Misclassifications (actual -> predicted, count):
  Fungal infection -> Chicken pox: 1


## 4. Feature importances — which symptoms matter most

Useful both for understanding the model and for picking which ~40-50 symptoms to actually
show as checkboxes in the Flutter UI (instead of all 132).

In [8]:
importances = pd.Series(clf.feature_importances_, index=symptom_columns)
top_symptoms = importances.sort_values(ascending=False)

print("Top 30 most important symptoms:")
top_symptoms.head(30)

Top 30 most important symptoms:


muscle_pain              0.019153
itching                  0.017178
nausea                   0.014942
mild_fever               0.014545
yellowing_of_eyes        0.014365
family_history           0.014295
high_fever               0.014022
altered_sensorium        0.014021
lack_of_concentration    0.013830
fatigue                  0.013750
sweating                 0.013182
vomiting                 0.013148
chest_pain               0.012742
joint_pain               0.012460
weight_loss              0.012274
unsteadiness             0.012176
dark_urine               0.011756
diarrhoea                0.011737
abdominal_pain           0.011427
red_spots_over_body      0.011255
muscle_weakness          0.010995
mucoid_sputum            0.010506
chills                   0.010495
bladder_discomfort       0.010388
rusty_sputum             0.010377
internal_itching         0.010104
headache                 0.010086
breathlessness           0.010054
pain_behind_the_eyes     0.009946
stomach_pain  

In [9]:
# Build the trimmed symptom list for the UI — top 50 by importance, sorted alphabetically
# for display (so the checkbox list isn't ordered by an opaque importance score)
UI_SYMPTOM_COUNT = 50
ui_symptoms = sorted(top_symptoms.head(UI_SYMPTOM_COUNT).index.tolist())

print(f"Selected {len(ui_symptoms)} symptoms for the UI checkbox list:")
print(ui_symptoms)

Selected 50 symptoms for the UI checkbox list:
['abdominal_pain', 'altered_sensorium', 'back_pain', 'bladder_discomfort', 'blood_in_sputum', 'breathlessness', 'chest_pain', 'chills', 'coma', 'continuous_feel_of_urine', 'cough', 'dark_urine', 'dehydration', 'diarrhoea', 'dischromic _patches', 'family_history', 'fast_heart_rate', 'fatigue', 'headache', 'high_fever', 'increased_appetite', 'internal_itching', 'itching', 'joint_pain', 'lack_of_concentration', 'loss_of_appetite', 'malaise', 'mild_fever', 'mucoid_sputum', 'muscle_pain', 'muscle_weakness', 'nausea', 'neck_pain', 'nodal_skin_eruptions', 'pain_behind_the_eyes', 'passage_of_gases', 'receiving_unsterile_injections', 'red_spots_over_body', 'rusty_sputum', 'spotting_ urination', 'stomach_bleeding', 'stomach_pain', 'sunken_eyes', 'sweating', 'unsteadiness', 'vomiting', 'weight_loss', 'yellow_crust_ooze', 'yellowing_of_eyes', 'yellowish_skin']


## 5. Cross-check predicted conditions against Nepal's Essential Drug List

This is the step that makes the model's output Nepal-relevant rather than a blind US/generic
dataset output. Manually build `dda_essential_conditions.json` by reading the DDA Nepal
Essential Drug List (https://dda.gov.np/category/required-medicine-list/) and noting which
broad condition categories it covers (e.g. respiratory infection, hypertension, diabetes,
GI infection, etc). This is a manual curation step — there is no API, so a team member reads
the published list and writes this mapping by hand. It only needs to be done once.

Below is a placeholder structure showing the expected shape. Replace with the real curated
list before this notebook is finalized.

In [10]:
# PLACEHOLDER -- replace with actual DDA Nepal Essential Drug List categories
# after manual review of https://dda.gov.np/category/required-medicine-list/
# Map: disease label (from the dataset) -> whether it has a realistic local treatment path
# This is intentionally left as a manual curation task, not an automated scrape, since the
# DDA list is published as PDF/HTML pages, not structured data.

dda_relevant_conditions = {
    # "Common Cold": True,
    # "Pneumonia": True,
    # "Malaria": True,
    # "Typhoid": True,
    # "Hypertension": True,
    # "Diabetes": True,
    # "GERD": True,
    # ... fill in after manual review
}

print("TODO: populate dda_relevant_conditions by reviewing the DDA Essential Drug List.")
print("Until populated, the Django integration should default every condition to")
print("'verify with the Nepal Essential Drug List' rather than silently assuming relevance.")

TODO: populate dda_relevant_conditions by reviewing the DDA Essential Drug List.
Until populated, the Django integration should default every condition to
'verify with the Nepal Essential Drug List' rather than silently assuming relevance.


## 6. Export for Django

In [11]:
joblib.dump(clf, OUTPUT_DIR / "symptom_checker_model.joblib")

export_metadata = {
    "all_symptom_columns": symptom_columns,
    "ui_symptom_columns": ui_symptoms,
    "disease_labels": sorted(y_train.unique().tolist()),
    "dda_relevant_conditions": dda_relevant_conditions,
}

with open(OUTPUT_DIR / "symptom_checker_metadata.json", "w") as f:
    json.dump(export_metadata, f, indent=2)

print("Saved:")
print(" -", OUTPUT_DIR / "symptom_checker_model.joblib")
print(" -", OUTPUT_DIR / "symptom_checker_metadata.json")
print()
print("Django loads symptom_checker_model.joblib once at startup. At request time, it")
print("builds a 132-length binary vector from the user's selected symptoms (in the exact")
print("order of all_symptom_columns), calls clf.predict(), and looks up the result against")
print("dda_relevant_conditions before surfacing it to the user.")

Saved:


 - ..\models\symptom_checker_model.joblib
 - ..\models\symptom_checker_metadata.json

Django loads symptom_checker_model.joblib once at startup. At request time, it
builds a 132-length binary vector from the user's selected symptoms (in the exact
order of all_symptom_columns), calls clf.predict(), and looks up the result against
dda_relevant_conditions before surfacing it to the user.
